In [2]:
import os

# Resolve code/beh_ephys_analysis (the folder containing `utils`) relative to this
# file's location, so imports work no matter where the repo is checked out.
import pathlib
import sys

# Resolve code/beh_ephys_analysis (the folder containing `utils`) relative to this
# file's location, so imports work no matter where the repo is checked out.
try:
    _here = pathlib.Path(__file__).resolve().parent          # script mode
except NameError:
    try:
        _here = pathlib.Path(__vsc_ipynb_file__).resolve().parent  # VS Code notebook
    except NameError:
        _here = pathlib.Path(os.getcwd())                    # other Jupyter fallback

_beh_ephys_root = str((_here / '../..').resolve())
if _beh_ephys_root not in sys.path:
    sys.path.insert(0, _beh_ephys_root)
import json
import pickle
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import pearsonr
from utils.beh_functions import (
    parseSessionID,
)
from utils.capsule_migration import capsule_directories
from utils.combine_tools import (
    apply_qc,
    to_str_intlike,
)
from utils.ephys_functions import *

warnings.filterwarnings('ignore')
capsule_dirs = capsule_directories()

%matplotlib inline
from utils.capsule_migration import CAPSULE_ROOT
from utils.panel_utils import columns_to_df, save_panel_csv, save_panels
from utils.beh_functions import get_unit_tbl


In [3]:
session = 'behavior_751004_2024-12-20_13-26-11'

In [4]:
from utils.beh_functions import session_dirs

# load unit table from data/scratch_data
unit_tbl = get_unit_tbl(session, 'curated')

# load LC range from scratch folder
caps = capsule_directories()
sd_curr = session_dirs(session, scratch_dir=caps['derived_dir_curr'])
lc_json = os.path.join(sd_curr['opto_dir_curated'], f'{session}_lc_range.json')
with open(lc_json) as _f:
    lc_range = json.load(_f).get('lc_range', {})
lc_top = lc_range.get('a')   # upper depth bound
lc_bot = lc_range.get('b')   # lower depth bound (None if not set)
print(f'LC range: bottom={lc_bot}, top={lc_top}')

# mask: units within LC depth range
in_lc = unit_tbl['y_loc'] <= lc_top
if lc_bot is not None:
    in_lc = in_lc & (unit_tbl['y_loc'] >= lc_bot)
unit_tbl['in_lc'] = in_lc
print(f'{in_lc.sum()} / {len(unit_tbl)} units in LC range')

LC range: bottom=150, top=800
70 / 248 units in LC range
